# 1. Requests

In [1]:
import requests

In [2]:
url = "https://books.toscrape.com/"
response = requests.get(url)

In [3]:
type(response)

requests.models.Response

In [4]:
response.__dict__.keys()

dict_keys(['_content', '_content_consumed', '_next', 'status_code', 'headers', 'raw', 'url', 'encoding', 'history', 'reason', 'cookies', 'elapsed', 'request', 'connection'])

In [5]:
# Посмотрим статус
print(response.status_code)

200


In [6]:
# Посмотрим html
print(response.text[:500])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


In [7]:
len(response.text)

51294

# 2. Анатомия HTTP запроса

- Метод: GET, POST
- URL: Адрес ресурса
- Заголовки (Headers):	Метаданные запроса
- Тело (Body):	Данные (для POST)
- Ответ (Response):	Статус-код + тело

<html>
  <body>
    <div class="product">
      <h3><a href="/book/1">Гарри Поттер</a></h3>
      <p class="price">₽1200</p>
    </div>
    <div class="product">
      <h3><a href="/book/2">Властелин колец</a></h3>
      <p class="price">₽950</p>
    </div>
  </body>
</html>

**Структура HTML**

- Тег (\<div\>, \<p\>, \<a\>) — элемент разметки.
- Атрибут (class="product", href="/book/1") — свойство тега.
- Дерево вложенности — каждый элемент может содержать дочерние элементы.
- CSS-селектор — способ адресации элементов по классу/тегу/иерархии, тот же синтаксис, что и в CSS-стилях.
- XPath — альтернативный язык адресации элементов через путь в дереве.

# 3. BeautifulSoup

### 3.1. Заголовок страницы

In [8]:
from bs4 import BeautifulSoup
import requests

In [9]:
url = "https://books.toscrape.com/"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

In [10]:
# Название страницы
soup.title

<title>
    All products | Books to Scrape - Sandbox
</title>

In [11]:
# Только текст
soup.title.get_text()

'\n    All products | Books to Scrape - Sandbox\n'

In [12]:
# Найти заголовок по тегу
title = soup.find("title")
print(title.get_text().strip())

All products | Books to Scrape - Sandbox


### 3.2. Структура страницы

In [13]:
books = soup.select("article.product_pod")
len(books)

20

In [14]:
print(books[0].prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   Â£51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



### 3.3. Получаем заголовок книги

In [15]:
book = books[0]
title = book.select_one("h3 a")
print(title.get_text())

A Light in the ...


In [16]:
type(title)

bs4.element.Tag

In [17]:
title

<a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a>

In [18]:
title["title"]

'A Light in the Attic'

In [19]:
# Другой вариант:
title = book.find("h3")
print(title.get_text())

A Light in the ...


In [20]:
type(title)

bs4.element.Tag

In [21]:
title.find("a")["title"]

'A Light in the Attic'

### 3.4. Получаем ссылку на книгу

In [22]:
from urllib.parse import urljoin

In [23]:
link = book.select_one("h3 a")
print(link["href"])

catalogue/a-light-in-the-attic_1000/index.html


In [24]:
url + link["href"]

'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'

In [25]:
book_url = urljoin(url, link["href"])
print(book_url)

https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html


### 3.5. Получаем цену

In [26]:
price = book.select_one(".price_color")
print(price.get_text())

Â£51.77


### 3.6. Собираем всю информацию о книге

In [27]:
def collect_book_data(book):
    title = book.select_one("h3 a")["title"]
    price = book.select_one(".price_color").get_text(strip=True)
    rating = book.select_one(".star-rating")["class"][1]
    availability = book.select_one(".availability").get_text(strip=True)
    book_url = urljoin(url, book.select_one("h3 a")["href"])
    
    book_data = {
        "title": title,
        "price": price,
        "rating": rating,
        "availability": availability,
        "url": book_url
    }
    
    return book_data

In [28]:
book = books[0]
collect_book_data(book)

{'title': 'A Light in the Attic',
 'price': 'Â£51.77',
 'rating': 'Three',
 'availability': 'In stock',
 'url': 'https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'}

### 3.7. Собираем информацию обо всех книгах

In [29]:
books_data = []
for book in books:
    books_data.append(collect_book_data(book))

In [30]:
len(books_data)

20

### 3.8. Трансформируем в DataFrame

In [31]:
import pandas as pd

In [32]:
df = pd.DataFrame(books_data)
df.head()

,title,price,rating,availability,url
0,A Light in the Attic,Â£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [33]:
df["price"] = df["price"].str.replace("Â£", "", regex=False).astype(float)
df.head()

,title,price,rating,availability,url
0,A Light in the Attic,51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


# 4. Переход по страницам

In [34]:
import time

In [35]:
all_books = []
for page in range(1, 6):
    url = f"https://books.toscrape.com/catalogue/page-{page}.html"

    response = requests.get(url, timeout=10) #timeout - время ожидания отклика

    response.raise_for_status() # Если status = 200, то молча продолжает. Если нет - останавливается и показывает статус и ошибку

    soup = BeautifulSoup(response.text,"html.parser")

    books = soup.select("article.product_pod")

    for book in books:
        all_books.append(collect_book_data(book))
    
    time.sleep(1) # Задержка между страницами

In [36]:
len(all_books)

100

In [37]:
df = pd.DataFrame(all_books)
df.head()

,title,price,rating,availability,url
0,A Light in the Attic,Â£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History of Humankind,Â£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...


In [38]:
df.tail()

,title,price,rating,availability,url
95,Lumberjanes Vol. 3: A Terrible Plan (Lumberjan...,Â£19.92,Two,In stock,https://books.toscrape.com/catalogue/lumberjan...
96,"Layered: Baking, Building, and Styling Spectac...",Â£40.11,One,In stock,https://books.toscrape.com/catalogue/layered-b...
97,Judo: Seven Steps to Black Belt (an Introducto...,Â£53.90,Two,In stock,https://books.toscrape.com/catalogue/judo-seve...
98,Join,Â£35.67,Five,In stock,https://books.toscrape.com/catalogue/join_902/...
99,In the Country We Love: My Family Divided,Â£22.00,Four,In stock,https://books.toscrape.com/catalogue/in-the-co...


# 5. Sessions

In [39]:
session = requests.Session()

In [40]:
response = session.get("https://books.toscrape.com/",timeout=10)
response.status_code

200

In [41]:
response = session.get("https://books.toscrape.com/catalogue/page-2.html",timeout=10)
response.status_code

200

# 6. Headers

In [42]:
session.headers.update({"User-Agent": "Mozilla/5.0"})

# 7. Cookies

In [43]:
session.cookies.set("my_cookie","hello")

Cookie(version=0, name='my_cookie', value='hello', port=None, port_specified=False, domain='', domain_specified=False, domain_initial_dot=False, path='/', path_specified=True, secure=False, expires=None, discard=True, comment=None, comment_url=None, rest={'HttpOnly': None}, rfc2109=False)

In [44]:
for cookie in session.cookies:
    print(cookie.name, cookie.value)

my_cookie hello


# 8. Проверка robots.txt

In [45]:
from urllib.robotparser import RobotFileParser

In [46]:
def can_fetch(url: str, user_agent: str = "*") -> bool:
    """Check whether scraping is allowed by robots.txt for the given URL."""
    rp = RobotFileParser()
    rp.set_url("https://www.forbes.ru/robots.txt")
    rp.read()
    return rp.can_fetch(user_agent, url)

In [47]:
print(can_fetch("https://www.forbes.ru/milliardery/463151-88-rossijskih-milliarderov-rejting-forbes-2022"))

True


# 9. Проверка задержки между запросами

In [48]:
# Проверка "задержки между запросами", если она задана в robots.txt (crawl-delay)
def check_delay(robots_url: str):
    rp = RobotFileParser()
    rp.set_url(robots_url)
    return rp.crawl_delay("MyCourseBot/1.0")

In [49]:
delay = check_delay("https://www.forbes.ru/robots.txt")
print(f"Рекомендуемая задержка между запросами: {delay} сек" if delay else "Задержка не указана явно")

Задержка не указана явно


# 10. Selenium

In [50]:
# pip install selenium

In [58]:
from selenium import webdriver
driver = webdriver.Chrome()

In [59]:
driver.get("https://quotes.toscrape.com/")

In [60]:
html = driver.page_source
print(html[:200])

<html lang="en"><head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
  


In [61]:
soup = BeautifulSoup(driver.page_source,"html.parser")

In [62]:
driver.quit()

# 11. Пример Selenium + bs4

In [55]:
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from bs4 import BeautifulSoup

In [56]:
driver = webdriver.Chrome()

try:

    # 1. Открываем JS-версию сайта
    driver.get("https://quotes.toscrape.com/js/")

    wait = WebDriverWait(driver,10)

    all_quotes = []

    # 2. Обрабатываем две страницы
    for page in range(1, 3):

        # 3. Ждем появления динамического контента
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".quote")))

        # 4. Получаем HTML после выполнения JavaScript
        html = driver.page_source

        # 5. Передаем HTML BeautifulSoup
        soup = BeautifulSoup(html,"html.parser")

        # 6. Извлекаем данные
        for quote in soup.select(".quote"):

            text = quote.select_one(".text").get_text(strip=True)

            author = quote.select_one(".author").get_text(strip=True)

            tags = [tag.get_text(strip=True) for tag in quote.select(".tag")]

            all_quotes.append({
                "page": page,
                "text": text,
                "author": author,
                "tags": tags
            })

        # 7. Нажимаем Next
        next_button = driver.find_element(
            By.CSS_SELECTOR,
            "li.next a"
        )

        next_button.click()

finally:

    # 8. Закрываем браузер
    driver.quit()


# 9. DataFrame
df = pd.DataFrame(all_quotes)
 
df.head()

,page,text,author,tags
0,1,“The world as we have created it is a process ...,Albert Einstein,"[change, deep-thoughts, thinking, world]"
1,1,"“It is our choices, Harry, that show what we t...",J.K. Rowling,"[abilities, choices]"
2,1,“There are only two ways to live your life. On...,Albert Einstein,"[inspirational, life, live, miracle, miracles]"
3,1,"“The person, be it gentleman or lady, who has ...",Jane Austen,"[aliteracy, books, classic, humor]"
4,1,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe,"[be-yourself, inspirational]"


# 12. Хорошие практики Scrapera

In [63]:
import time
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

In [64]:
retry_strategy = Retry(
    total=3,
    backoff_factor=1,
    status_forcelist=[
        429,
        500,
        502,
        503,
        504
    ],
    allowed_methods=["GET"]
)


In [65]:
session = requests.Session()

adapter = HTTPAdapter(
    max_retries=retry_strategy
)

session.mount(
    "https://",
    adapter
)

session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

In [66]:
def get_page(url):

    response = session.get(
        url,
        timeout=10
    )

    response.raise_for_status()

    return response.text

In [67]:
def parse_books(html, page_url):

    soup = BeautifulSoup(
        html,
        "html.parser"
    )

    books = []

    for book in soup.select(
        "article.product_pod"
    ):

        title = book.select_one(
            "h3 a"
        ).get_text(strip=True)

        price = book.select_one(
            ".price_color"
        ).get_text(strip=True)

        rating = book.select_one(
            ".star-rating"
        )["class"][1]

        availability = book.select_one(
            ".availability"
        ).get_text(strip=True)

        link = book.select_one(
            "h3 a"
        )["href"]

        book_url = urljoin(
            page_url,
            link
        )

        books.append({
            "title": title,
            "price": price,
            "rating": rating,
            "availability": availability,
            "url": book_url
        })

    return books

In [68]:
all_books = []

for page in range(1, 6):

    url = (
        "https://books.toscrape.com/"
        f"catalogue/page-{page}.html"
    )

    print(
        f"Обрабатываем страницу {page}"
    )

    html = get_page(url)

    books = parse_books(
        html,
        url
    )

    all_books.extend(books)

    time.sleep(1)

Обрабатываем страницу 1
Обрабатываем страницу 2
Обрабатываем страницу 3
Обрабатываем страницу 4
Обрабатываем страницу 5


In [69]:
df = pd.DataFrame(all_books)
df.head()

,title,price,rating,availability,url
0,A Light in the ...,Â£51.77,Three,In stock,https://books.toscrape.com/catalogue/a-light-i...
1,Tipping the Velvet,Â£53.74,One,In stock,https://books.toscrape.com/catalogue/tipping-t...
2,Soumission,Â£50.10,One,In stock,https://books.toscrape.com/catalogue/soumissio...
3,Sharp Objects,Â£47.82,Four,In stock,https://books.toscrape.com/catalogue/sharp-obj...
4,Sapiens: A Brief History ...,Â£54.23,Five,In stock,https://books.toscrape.com/catalogue/sapiens-a...
